In [1]:
import numpy as np

word = "dogs"
chars = list(set(word)) # ['d', 'o', 'g', 's']
vocab_size = len(chars) # V = 4

char_to_ix = { ch:i for i,ch in enumerate(chars) }
ix_to_char = { i:ch for i,ch in enumerate(chars) }

print(f"Mappings: {char_to_ix}")

Mappings: {'d': 0, 'g': 1, 's': 2, 'o': 3}


In [2]:
hidden_size = 3  # d = 3
learning_rate = 0.1

Wxh = np.random.randn(hidden_size, vocab_size) * 0.01   # (3, 4)
Whh = np.random.randn(hidden_size, hidden_size) * 0.01  # (3, 3)
Why = np.random.randn(vocab_size, hidden_size) * 0.01   # (4, 3)

bh = np.zeros((hidden_size, 1)) # (3, 1)
by = np.zeros((vocab_size, 1))  # (4, 1)

In [3]:
def to_one_hot(index, size):
    x = np.zeros((size, 1))
    x[index] = 1
    return x

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

In [4]:
def forward_backward(inputs, targets, hprev):
    xs, hs, ys, ps = {}, {}, {}, {}
    hs[-1] = np.copy(hprev)
    loss = 0

    # ---> Forward Pass <---
    for t in range(len(inputs)):
        xs[t] = to_one_hot(inputs[t], vocab_size)
        # h_t = tanh(Wxh * x_t + Whh * h_{t-1} + bh)
        hs[t] = np.tanh(np.dot(Wxh, xs[t]) + np.dot(Whh, hs[t-1]) + bh)
        ys[t] = np.dot(Why, hs[t]) + by
        ps[t] = softmax(ys[t])
        loss += -np.log(ps[t][targets[t], 0])

    # ---> Backward Pass (BPTT) <---
    dWxh, dWhh, dWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
    dbh, dby = np.zeros_like(bh), np.zeros_like(by)
    dhnext = np.zeros_like(hs[0])

    for t in reversed(range(len(inputs))):
        dy = np.copy(ps[t])
        dy[targets[t]] -= 1
        dWhy += np.dot(dy, hs[t].T)
        dby += dy
        dh = np.dot(Why.T, dy) + dhnext
        dhraw = (1 - hs[t] * hs[t]) * dh
        dbh += dhraw
        dWxh += np.dot(dhraw, xs[t].T)
        dWhh += np.dot(dhraw, hs[t-1].T)
        dhnext = np.dot(Whh.T, dhraw)

    return loss, dWxh, dWhh, dWhy, dbh, dby, hs[len(inputs)-1]

In [5]:
input_idxs = [char_to_ix[ch] for ch in "dog"]
target_idxs = [char_to_ix[ch] for ch in "ogs"]
hprev = np.zeros((hidden_size, 1))

print("Training started...")
for i in range(1001):
    loss, dWxh, dWhh, dWhy, dbh, dby, hlast = forward_backward(input_idxs, target_idxs, hprev)

    for param, dparam in zip([Wxh, Whh, Why, bh, by], [dWxh, dWhh, dWhy, dbh, dby]):
        param -= learning_rate * dparam

    if i % 200 == 0:
        print(f"Iteration {i}, Loss: {loss:.4f}")

Training started...
Iteration 0, Loss: 4.1588
Iteration 200, Loss: 0.0415
Iteration 400, Loss: 0.0170
Iteration 600, Loss: 0.0106
Iteration 800, Loss: 0.0077
Iteration 1000, Loss: 0.0061


In [6]:
def predict(word_in):
    h = np.zeros((hidden_size, 1))
    for ch in word_in:
        x = to_one_hot(char_to_ix[ch], vocab_size)
        h = np.tanh(np.dot(Wxh, x) + np.dot(Whh, h) + bh)

    y = np.dot(Why, h) + by
    p = softmax(y)
    return ix_to_char[np.argmax(p)]

result = predict("dog")
print(f"Input: 'dog' -> Predicted next letter: '{result}'")

Input: 'dog' -> Predicted next letter: 's'
